# 🔁 Every Session — Run These Cells First
### Run this notebook at the start of every Colab session.
---

## Step 1 — Install Packages
Colab wipes packages on every session disconnect.

In [1]:
import torch
print(torch.cuda.is_available())      # must say True
print(torch.cuda.get_device_name(0))  # must say Tesla T4

False


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
!pip install -q transformers peft bitsandbytes trl accelerate torch
print("✅ Packages ready")

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")

## Step 3 — Copy Model from Drive to Local Disk
Drive I/O is slow. Copying to `/content/` makes loading much faster.
Takes ~2-5 minutes.

In [ ]:
import os

if not os.path.exists("/content/model"):
    print("Copying model to local disk...")
    !cp -r "/content/drive/MyDrive/Qwen2.5-3B-Instruct" "/content/model"
    print("✅ Model copied to /content/model")
else:
    print("✅ Model already on local disk, skipping copy")

MODEL_PATH = "/content/model"

## Step 4 — Load Model & Tokenizer

In [ ]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=True
)
print("✅ Tokenizer loaded")

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,    
    device_map="auto",
    local_files_only=True,
    trust_remote_code=True,
    low_cpu_mem_usage=True          
)

print("✅ Model loaded")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
import torch
print(torch.cuda.is_available())      # if False → GPU not enabled
print(torch.cuda.get_device_name(0))  # if error → no GPU

## Step 5 — Quick Test (Optional)
Confirm the model is working before you start training.

In [ ]:
messages = [{"role": "user", "content": "Who are you?"}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

---
## ✅ Ready!
Model is loaded. Continue below with your fine-tuning or inference code.

**Remember:** Save any outputs/checkpoints to `/content/drive/MyDrive/` so they persist after the session ends.

---
## Fine-Tuning (when ready)
Add your training code below this cell.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for LoRA fine-tuning
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Add your dataset and SFTTrainer below ──
# OUTPUT_DIR = "/content/drive/MyDrive/Qwen2.5-3B-finetuned"  # saves to Drive